# T5-base Fine-Tuning on Spider (Full Run)

Setup for Colab Pro (L4 / A100):
- **T5-base** (220M parameters)
- **Full training data**: `train_spider.json` + `train_others.json` (~8600 examples)
- **8 epochs** with load-best-by-eval-loss
- **Input 768 tokens**: fits virtually all schemas without truncation
- **Rich schema serialization** with tables, column types, foreign keys
- **Checkpoint-resume** from Drive if session dies

**Runtime**: ~1.5-2h on L4, ~60-90 min on A100.

**Before running**:
- Runtime → Change runtime type → GPU → L4 (or A100 if available)
- Upload to Drive under `MyDrive/text-to-sql/spider_data/`:
  - `train_spider.json`, `train_others.json`, `dev.json`, `tables.json`

**If Colab disconnects mid-training**: re-run the notebook. It resumes from the latest Drive checkpoint.

## 1. Setup

In [ ]:
!pip install -q "transformers>=4.40" "datasets>=2.14" "accelerate>=0.30"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
import random
import numpy as np
from pathlib import Path

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

DRIVE_ROOT = Path('/content/drive/MyDrive/text-to-sql')
SPIDER_DIR = DRIVE_ROOT / 'spider_data'
CHECKPOINT_DIR = DRIVE_ROOT / 't5_base_spider'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

for f in ['train_spider.json', 'train_others.json', 'dev.json', 'tables.json']:
    assert (SPIDER_DIR / f).exists(), f'Missing: {SPIDER_DIR / f}'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('No GPU. Change Runtime → GPU.')

## 2. Load Spider & build schema strings

Rich schema serialization: tables with columns and column types, followed by foreign key relationships. The FKs help T5 learn correct joins across tables.

In [ ]:
import json

def load_json(path):
    with open(path) as f:
        return json.load(f)

train = load_json(SPIDER_DIR / 'train_spider.json')
train_others = load_json(SPIDER_DIR / 'train_others.json')
validation = load_json(SPIDER_DIR / 'dev.json')
tables = load_json(SPIDER_DIR / 'tables.json')

print(f'train_spider: {len(train)}')
print(f'train_others: {len(train_others)}')
print(f'validation: {len(validation)}')
print(f'schemas: {len(tables)}')

In [ ]:
def build_schema_string(db):
    table_names = db['table_names_original']
    columns = db['column_names_original']
    column_types = db['column_types']
    foreign_keys = db.get('foreign_keys', [])

    tables_to_cols = {i: [] for i in range(len(table_names))}
    col_idx_to_ref = {}
    for col_idx, (tbl_idx, col_name) in enumerate(columns):
        if tbl_idx == -1:
            continue
        col_type = column_types[col_idx]
        tables_to_cols[tbl_idx].append(f'{col_name}:{col_type}')
        col_idx_to_ref[col_idx] = f'{table_names[tbl_idx]}.{col_name}'

    table_parts = []
    for i, tbl in enumerate(table_names):
        cols = ', '.join(tables_to_cols[i])
        table_parts.append(f'{tbl}({cols})')

    fk_parts = []
    for a, b in foreign_keys:
        if a in col_idx_to_ref and b in col_idx_to_ref:
            fk_parts.append(f'{col_idx_to_ref[a]} = {col_idx_to_ref[b]}')

    schema = ' | '.join(table_parts)
    if fk_parts:
        schema += ' | foreign_keys: ' + ', '.join(fk_parts)
    return schema

schema_by_db = {t['db_id']: build_schema_string(t) for t in tables}
print('Sample schema (concert_singer):')
print(schema_by_db['concert_singer'])

In [ ]:
def make_input(ex):
    return f"translate English to SQL | {ex['db_id']} | {ex['question']} | schema: {schema_by_db[ex['db_id']]}"

def clean_target(sql):
    return ' '.join(sql.split())

def to_t5_pairs(examples):
    return [
        {'input': make_input(ex), 'target': clean_target(ex['query'])}
        for ex in examples
    ]

train_pairs = to_t5_pairs(train) + to_t5_pairs(train_others)
val_pairs = to_t5_pairs(validation)

print(f'train pairs: {len(train_pairs)}')
print(f'val pairs: {len(val_pairs)}')
print()
print('sample input:')
print(train_pairs[0]['input'])
print()
print('sample target:', train_pairs[0]['target'])

## 3. Tokenize

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = 't5-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_INPUT_LEN = 768
MAX_TARGET_LEN = 256

input_lens = [len(tokenizer.encode(p['input'])) for p in train_pairs[:500]]
target_lens = [len(tokenizer.encode(p['target'])) for p in train_pairs[:500]]
print(f'Input tokens — p50: {np.percentile(input_lens, 50):.0f}, p95: {np.percentile(input_lens, 95):.0f}, max: {max(input_lens)}')
print(f'Target tokens — p50: {np.percentile(target_lens, 50):.0f}, p95: {np.percentile(target_lens, 95):.0f}, max: {max(target_lens)}')

In [ ]:
from datasets import Dataset

def tokenize_fn(batch):
    model_inputs = tokenizer(
        batch['input'], max_length=MAX_INPUT_LEN, truncation=True
    )
    labels = tokenizer(
        text_target=batch['target'], max_length=MAX_TARGET_LEN, truncation=True
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

train_ds = Dataset.from_list(train_pairs).map(
    tokenize_fn, batched=True, remove_columns=['input', 'target']
)
val_ds = Dataset.from_list(val_pairs).map(
    tokenize_fn, batched=True, remove_columns=['input', 'target']
)

print(train_ds)
print(val_ds)

## 4. Train

- 8 epochs, batch 16, LR 1e-4 (T5's typical sweet spot)
- fp16 on L4/A100 saturates the GPU well
- Save every epoch, keep best 2, load best by eval_loss at the end
- If VRAM complains, drop `per_device_train_batch_size` to 8 and set `gradient_accumulation_steps=2`

In [ ]:
from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
collator = DataCollatorForSeq2Seq(tokenizer, model=model, pad_to_multiple_of=8)

args = Seq2SeqTrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type='linear',
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    predict_with_generate=False,
    fp16=True,
    report_to='none',
    seed=SEED,
    dataloader_num_workers=2,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=collator,
)

In [ ]:
existing_ckpts = sorted(CHECKPOINT_DIR.glob('checkpoint-*'))
resume = len(existing_ckpts) > 0

if resume:
    print(f'Resuming from {len(existing_ckpts)} existing checkpoint(s):')
    for c in existing_ckpts:
        print(f'  {c.name}')
else:
    print('Starting fresh training run')

trainer.train(resume_from_checkpoint=resume)

In [ ]:
FINAL_DIR = CHECKPOINT_DIR / 'final'
trainer.save_model(str(FINAL_DIR))
tokenizer.save_pretrained(str(FINAL_DIR))
print(f'Best model (by eval loss) saved to {FINAL_DIR}')
print()
print('Eval loss per epoch:')
for log in trainer.state.log_history:
    if 'eval_loss' in log:
        print(f"  epoch {log['epoch']:.1f}: eval_loss = {log['eval_loss']:.4f}")

## 5. Sanity check

Quick look at a few predictions to confirm the model learned something before running full inference.

In [ ]:
model.eval()
for ex in val_pairs[:5]:
    inputs = tokenizer(
        ex['input'], return_tensors='pt', truncation=True, max_length=MAX_INPUT_LEN
    ).to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=MAX_TARGET_LEN, num_beams=4, early_stopping=True
        )
    pred = tokenizer.decode(out[0], skip_special_tokens=True)
    print('GOLD:', ex['target'])
    print('PRED:', pred)
    print()

## 6. Cleanup intermediate checkpoints (optional)

After `final/` is saved, you can drop the intermediate `checkpoint-XXXX/` folders from Drive to free space.

In [ ]:
import shutil

DELETE_CHECKPOINTS = False

if DELETE_CHECKPOINTS:
    for ckpt in CHECKPOINT_DIR.glob('checkpoint-*'):
        print(f'Deleting {ckpt}')
        shutil.rmtree(ckpt)
    print('Done')
else:
    print('Set DELETE_CHECKPOINTS = True and re-run this cell to clean up')